# 하나의 행렬로 연결하는 선형 변환, rank, 연립방정식

- 기준 TIL: [2026-08-18](../../til/2026/08/2026-08-18.md)
- 관련 강의자료: [02-01 선형 변환](../../materials/private/kant-basic-math/02-01_선형_변환의_기하학적_해석.md) · [02-02 벡터공간과 선형 독립](../../materials/private/kant-basic-math/02-02_벡터공간과_선형_독립.md) · [02-03 연립선형방정식](../../materials/private/kant-basic-math/02-03_연립선형방정식과_행렬_해법.md)
- 강의 제공 실습: [02-01](../../materials/private/kant-basic-math/course-provided-practice/02-01_선형_변환의_기하학적_해석.md) · [02-02](../../materials/private/kant-basic-math/course-provided-practice/02-02_벡터공간_선형독립_Rank.md) · [02-03](../../materials/private/kant-basic-math/course-provided-practice/02-03_연립방정식과_행렬_해법.md)
- 난이도: Core
- 상태: 완료


## 강의 제공 실습을 어떻게 다듬었는가

- `02-01`에서 실행 전 예측과 열벡터 관점의 변환 적용을 유지했다.
- `02-02`에서 rank 비교와 0이 아닌 영공간 벡터 구성을 유지했다.
- `02-03`에서 확대행렬을 이용한 해 분류와 `lstsq` 검산을 유지했다.
- 세 실습의 UCI 데이터 로드, 반복되는 전처리, 완성 답안과 예시 출력은 제외했다. 대신 작은 행렬을 계속 재사용해 변환의 정보 손실, rank, 영공간, 해의 종류가 같은 구조를 설명한다는 점을 확인한다.
- 정규방정식은 열이 종속이면 유일하게 풀리지 않을 수 있으므로, 마지막에는 이를 성공한다고 가정하지 않고 실패 가능성을 직접 관찰한다.


## 왜 지금 이 실습을 하는가

- TIL과 학습 답변에서 확인된 이해: 선형성으로 기저벡터의 상을 조합했고, 종속 벡터가 rank를 늘리지 않는다고 설명했다. `b`의 열공간 포함 여부로 무한해와 해 없음을 구분했으며, 최소제곱 잔차가 열공간과 직교함을 손으로 계산했다.
- 이번에 확인할 부족한 부분: 이 연결을 NumPy 배열의 Shape, 열을 묶는 방법, `matrix_rank`, 확대행렬, `lstsq` 반환값과 실제 수치 오차로 확인한 실행 기록은 없다.
- 핵심 질문: 한 행렬의 열벡터가 변환 결과, rank와 영공간, `Ax=b`의 해 종류를 어떻게 동시에 결정하며, 최소제곱 결과는 잔차의 직교성으로 어떻게 검산할 수 있는가?


## 목표와 완료 기준

- [x] 실행 전에 주요 Shape, rank, 해의 종류를 먼저 예상했다.
- [x] `A @ x`를 행렬의 열벡터 선형결합으로 다시 계산해 같은 결과인지 확인했다.
- [x] 0이 아닌 영공간 벡터를 직접 만들고 정보 손실과 연결했다.
- [x] 확대행렬의 rank로 유일해·무한해·해 없음을 코드로 분류했다.
- [x] `lstsq`의 계수, 잔차, rank, 특이값을 구분하고 `A.T @ r`로 최적성을 검산했다.
- [x] 열이 종속일 때 정규방정식과 계수의 유일성에 무엇이 달라지는지 관찰했다.
- [x] 이 작은 정확한 행렬 실험의 한계를 하나 적었다.


## 실행 전 예상

준비 셀의 숫자를 읽은 뒤, 이후 계산 셀을 실행하기 전에 먼저 작성한다.

1. `A_full`, `A_collapse`, `points`의 Shape은 각각 무엇인가?
    - (2, 2), (2, 2), (3, 2)
2. `A_full`의 첫째·둘째 열은 `e1`, `e2`를 각각 어디로 보내는가? 이 열들만 사용해 `A_full @ x_combo`를 예상한다.
    - [2, 0], [1, 1] / [5, -1]
3. 두 행렬의 rank는 각각 얼마라고 예상하는가?
    - 2, 1
4. `A_collapse @ c = 0`을 만족하는 0이 아닌 `c` 하나를 손으로 찾는다.
    - (-2, 1)
5. 같은 `A_collapse`에서 `b_in`, `b_out`은 각각 열공간 안과 밖 중 어디에 있는가? 해의 종류를 예상한다.
    - b_in은 열공간 안에 있고, b_out은 밖에 있다. / 안에 있다면 해는 무한대, 밖에 있으면 해없음
6. `A_ls @ coef`가 세 관측값에 가장 가까운 하나의 상수가 되려면 `coef`는 대략 얼마여야 하는가?
    - 5


In [1]:
# 준비: 위 예상부터 작성한 뒤 실행하세요.
import numpy as np

np.set_printoptions(precision=6, suppress=True)

e1 = np.array([1.0, 0.0])
e2 = np.array([0.0, 1.0])

A_full = np.array([[2.0, 1.0],
                   [0.0, 1.0]])
A_collapse = np.array([[1.0, 2.0],
                       [2.0, 4.0]])

x_combo = np.array([3.0, -1.0])
points = np.array([[1.0, 0.0],
                   [0.0, 1.0],
                   [3.0, -1.0]])

b_unique = np.array([5.0, 1.0])
b_in = np.array([3.0, 6.0])
b_out = np.array([3.0, 7.0])

A_ls = np.ones((3, 1))
b_ls = np.array([2.0, 4.0, 9.0])

print('A_full.shape:', A_full.shape)
print('A_collapse.shape:', A_collapse.shape)
print('points.shape:', points.shape)
print('A_ls.shape:', A_ls.shape, '/ b_ls.shape:', b_ls.shape)


A_full.shape: (2, 2)
A_collapse.shape: (2, 2)
points.shape: (3, 2)
A_ls.shape: (3, 1) / b_ls.shape: (3,)


## 1. 행렬의 열로 변환 결과 다시 만들기

1. `A_full @ e1`, `A_full @ e2`를 계산해 행렬의 두 열과 비교한다.
2. `A_full @ x_combo`를 직접 계산한다.
3. `x_combo`의 두 성분을 계수로 사용해 두 열의 선형결합을 만들고 직접 계산 결과와 비교한다.
4. `points`는 샘플이 행으로 쌓인 `(samples, features)` 배열이다. 각 행에 열벡터 관점의 `A_full @ v`를 적용하도록 배치 계산을 완성한다.
5. 배치 결과의 각 행이 개별 계산과 일치하는지 확인한다.


In [2]:
# TODO 1-A: 기저벡터의 상을 계산하세요.
image_e1 = A_full @ e1
image_e2 = A_full @ e2

# TODO 1-B: 직접 행렬곱과 열벡터 선형결합을 각각 계산하세요.
direct = A_full @ x_combo
from_columns = (
    x_combo[0] * A_full[:, 0]
    + x_combo[1] * A_full[:, 1]
)

# TODO 1-C: 행으로 쌓인 points에 같은 변환을 적용하세요.
points_transformed = points @ A_full.T

assert all(value is not None for value in
           (image_e1, image_e2, direct, from_columns, points_transformed))

print('T(e1):', image_e1)
print('T(e2):', image_e2)
print('직접 계산:', direct)
print('열 조합   :', from_columns)
print('두 계산 일치:', np.allclose(direct, from_columns))
print('배치 결과 shape:', points_transformed.shape)
print('마지막 행과 직접 계산 일치:', np.allclose(points_transformed[-1], direct))


T(e1): [2. 0.]
T(e2): [1. 1.]
직접 계산: [ 5. -1.]
열 조합   : [ 5. -1.]
두 계산 일치: True
배치 결과 shape: (3, 2)
마지막 행과 직접 계산 일치: True


### 1번 결과 해석

- `A_full`의 각 열과 `T(e1)`, `T(e2)`는 어떻게 대응했는가?
- `direct`와 `from_columns`가 같다는 결과가 선형성에 관해 무엇을 보여주는가?
- 열벡터 하나에는 `A @ v`를 쓰면서, 행으로 쌓인 배치에는 전치가 필요한 이유를 Shape으로 설명한다.


## 2. rank와 영공간으로 정보 손실 확인하기

1. `A_full`, `A_collapse`의 rank를 계산한다.
2. `A_collapse @ null_candidate = 0`을 만족하는 0이 아닌 길이 2 벡터를 직접 구성한다.
3. 두 행렬의 `열 수 - rank`를 계산하고, 0이 아닌 영공간 방향의 존재와 비교한다.
4. `null_candidate`만큼 차이 나는 두 입력이 왜 같은 출력으로 가는지 실제 계산으로 확인한다.


In [3]:
# TODO 2-A: 두 행렬의 rank를 계산하세요.
rank_full = np.linalg.matrix_rank(A_full)
rank_collapse = np.linalg.matrix_rank(A_collapse)

# TODO 2-B: 0이 아닌 영공간 벡터를 직접 구성하세요.
null_candidate = np.array([-2, 1])

# TODO 2-C: null_candidate만큼 차이 나는 두 입력을 만들고 출력을 비교하세요.
x_base = np.array([1.0, 1.0])
x_shifted = x_base + null_candidate
y_base = A_collapse @ x_base
y_shifted = A_collapse @ x_shifted

assert rank_full is not None and rank_collapse is not None
assert null_candidate is not None and x_shifted is not None
assert y_base is not None and y_shifted is not None

print('rank(A_full):', rank_full)
print('rank(A_collapse):', rank_collapse)
print('영공간 검산 A_collapse @ c:', A_collapse @ null_candidate)
print('영공간 벡터의 노름:', np.linalg.norm(null_candidate))
print('두 입력의 출력이 같은가:', np.allclose(y_base, y_shifted))
print('A_full 영공간 차원:', A_full.shape[1] - rank_full)
print('A_collapse 영공간 차원:', A_collapse.shape[1] - rank_collapse)


rank(A_full): 2
rank(A_collapse): 1
영공간 검산 A_collapse @ c: [0. 0.]
영공간 벡터의 노름: 2.23606797749979
두 입력의 출력이 같은가: True
A_full 영공간 차원: 0
A_collapse 영공간 차원: 1


### 2번 결과 해석

- `A_collapse`에서 열은 두 개지만 독립 방향은 몇 개인가?
- 서로 다른 두 입력이 같은 출력으로 간 결과가 역변환 가능성에 대해 무엇을 보여주는가?
- `열 수 = rank + 영공간 차원`이 이번 두 행렬에서 각각 어떻게 확인됐는가?


## 3. 확대행렬의 rank로 해의 종류 분류하기

`classify_system(A, b)`를 완성한다. 반환값에는 `rank_A`, `rank_augmented`, `n_variables`, `kind`를 넣는다.

판정 순서는 다음 관계를 코드로 옮긴다.

1. `rank(A)`와 `rank([A|b])`가 다르면 해 없음
2. 두 rank가 같고 변수 수와 같으면 유일해
3. 두 rank가 같지만 변수 수보다 작으면 무한해

세 시스템을 분류한 뒤 유일해 시스템만 `np.linalg.solve`로 풀고 `A @ x`로 검산한다.


In [4]:
def classify_system(A, b):
    # TODO 3-A: A와 확대행렬의 rank, 변수 수를 계산하세요.
    rank_A = np.linalg.matrix_rank(A)
    augmented = np.column_stack([A, b])
    rank_augmented = np.linalg.matrix_rank(augmented)
    n_variables = A.shape[1]

    kind = None

    # TODO 3-B: 세 rank 조건을 이용해 kind를 정하세요.
    if rank_A != rank_augmented:
        kind = 'no solution'
    elif rank_A == n_variables:
        kind = 'unique solution'
    else:
        kind = 'infinitely many solutions'


    return {
        'rank_A': rank_A,
        'rank_augmented': rank_augmented,
        'n_variables': n_variables,
        'kind': kind,
    }


systems = {
    'case_1': (A_full, b_unique),
    'case_2': (A_collapse, b_in),
    'case_3': (A_collapse, b_out),
}

# TODO 3-C: 각 시스템을 분류해 출력하세요.
classifications = {
    name: classify_system(A, b)
    for name, (A, b) in systems.items()
}

assert classifications is not None
print(classifications)

# TODO 3-D: 유일해인 시스템만 solve로 풀고 A @ x == b인지 검산하세요.

for name, (A, b) in systems.items():
    if classifications[name]["kind"] != "unique solution":
        continue

    unique_x = np.linalg.solve(A, b)
    unique_check = np.allclose(A @ unique_x, b)
    assert unique_x is not None and unique_check is not None
    print('유일해 계수:', unique_x)
    print('검산 결과:', unique_check)


{'case_1': {'rank_A': np.int64(2), 'rank_augmented': np.int64(2), 'n_variables': 2, 'kind': 'unique solution'}, 'case_2': {'rank_A': np.int64(1), 'rank_augmented': np.int64(1), 'n_variables': 2, 'kind': 'infinitely many solutions'}, 'case_3': {'rank_A': np.int64(1), 'rank_augmented': np.int64(2), 'n_variables': 2, 'kind': 'no solution'}}
유일해 계수: [2. 1.]
검산 결과: True


### 3번 결과 해석

- `case_2`와 `case_3`는 같은 `A`를 쓰는데 왜 해의 종류가 달랐는가?
- `rank([A|b])`가 증가한다는 것은 `b`와 기존 열공간의 관계에 대해 무엇을 뜻하는가?
- `solve`의 성공·실패만으로 무한해와 해 없음을 구분할 수 없는 이유를 적는다.


## 4. 최소제곱 반환값과 잔차 직교성 확인하기

1. `np.linalg.lstsq(A_ls, b_ls, rcond=None)`의 네 반환값을 빠짐없이 받는다.
2. `prediction = A_ls @ coef`와 `r = b_ls - prediction`을 계산한다.
3. 직접 계산한 잔차 제곱합과 `residuals`가 같은 내용을 나타내는지 비교한다.
4. `A_ls.T @ r`이 0에 가까운지 확인하고, 정확히 0이 아닐 수 있는 이유를 적는다.
5. 각 반환값의 Shape과 의미를 실제 출력에 연결한다.


In [5]:
# TODO 4-A: lstsq의 네 반환값을 받으세요.
coef, residuals, rank_ls, singular_values = np.linalg.lstsq(A_ls, b_ls, rcond=None)

# TODO 4-B: 예측, 잔차, 잔차 제곱합, 직교성 검산값을 계산하세요.
prediction = A_ls @ coef
r = b_ls - prediction
residual_sum_squares = np.sum(r ** 2)
orthogonality_check = A_ls.T @ r

assert all(value is not None for value in
           (coef, residuals, rank_ls, singular_values, prediction, r,
            residual_sum_squares, orthogonality_check))

print('coef / shape:', coef, coef.shape)
print('prediction / shape:', prediction, prediction.shape)
print('r / shape:', r, r.shape)
print('residuals:', residuals)
print('직접 계산한 잔차 제곱합:', residual_sum_squares)
print('rank:', rank_ls)
print('singular_values:', singular_values)
print('A_ls.T @ r:', orthogonality_check)
print('직교 조건을 수치적으로 만족:', np.allclose(orthogonality_check, 0.0))


coef / shape: [5.] (1,)
prediction / shape: [5. 5. 5.] (3,)
r / shape: [-3. -1.  4.] (3,)
residuals: [26.]
직접 계산한 잔차 제곱합: 26.0
rank: 1
singular_values: [1.732051]
A_ls.T @ r: [0.]
직교 조건을 수치적으로 만족: True


## 5. 열이 종속일 때 계수의 유일성 확인하기

이번에는 `A_ls`의 열과 그 2배인 열을 나란히 둔 `A_dep`을 사용한다.

1. `A_dep`의 Shape, rank, 영공간 차원을 예측하고 계산한다.
2. `lstsq`로 계수 하나를 구한다.
3. `A_dep @ null_shift = 0`인 0이 아닌 `null_shift`를 직접 만들어, `coef_dep + null_shift`도 같은 예측을 만드는지 확인한다.
4. `A_dep.T @ A_dep`로 만든 정규방정식을 `solve`로 시도하고 성공 여부 또는 오류를 기록한다.
5. 예측과 최소 잔차는 정해질 수 있는데 계수는 왜 하나로 정해지지 않는지 설명한다.


In [6]:
A_dep = np.column_stack([A_ls[:, 0], 2.0 * A_ls[:, 0]])

# TODO 5-A: rank와 lstsq 계수를 계산하세요.
rank_dep = np.linalg.matrix_rank(A_dep)
coef_dep, residual_dep, rank_lstsq_dep, singular_values_dep = np.linalg.lstsq(A_dep, b_ls, rcond=None)

# TODO 5-B: 0이 아닌 영공간 이동량을 구성하세요.
null_shift = np.array([2.0, -1.0])
shifted_coef = coef_dep + null_shift
same_prediction = np.allclose(
    A_dep @ coef_dep,
    A_dep @ shifted_coef
)

assert rank_dep is not None and coef_dep is not None
assert null_shift is not None and shifted_coef is not None
assert same_prediction is not None

print('A_dep.shape:', A_dep.shape)
print('rank / 영공간 차원:', rank_dep, '/', A_dep.shape[1] - rank_dep)
print('영공간 검산:', A_dep @ null_shift)
print('두 계수의 예측이 같은가:', same_prediction)

# TODO 5-C: 정규방정식을 solve로 시도하고 결과 또는 오류를 기록하세요.
normal_matrix = A_dep.T @ A_dep
normal_target = A_dep.T @ b_ls

try:
    normal_equation_status = np.linalg.solve(normal_matrix, normal_target)
except np.linalg.LinAlgError as error:
    normal_equation_status = f'failed: {error}'
assert normal_equation_status is not None
print('정규방정식 solve 결과:', normal_equation_status)


A_dep.shape: (3, 2)
rank / 영공간 차원: 1 / 1
영공간 검산: [0. 0. 0.]
두 계수의 예측이 같은가: True
정규방정식 solve 결과: failed: Singular matrix


## 막혔을 때 단계별 힌트

<details>
<summary>힌트 1: 행렬의 열과 기저벡터</summary>

`A @ e1`은 `A`의 첫 번째 열을, `A @ e2`는 두 번째 열을 선택한다. `x_combo`의 성분을 그 두 열에 붙는 계수로 읽는다.
</details>

<details>
<summary>힌트 2: 행으로 쌓인 배치</summary>

개별 열벡터에는 `A @ v`를 쓴다. 각 샘플이 행인 `(samples, features)` 배열에서는 안쪽 Shape과 결과 방향을 맞추기 위해 어느 행렬을 전치해야 하는지 적어 본다.
</details>

<details>
<summary>힌트 3: 영공간 벡터</summary>

`A_collapse`의 둘째 열이 첫째 열의 몇 배인지 찾는다. 두 열을 더했을 때 0이 되도록 서로 반대 부호의 계수를 둔다.
</details>

<details>
<summary>힌트 4: 확대행렬과 판정 순서</summary>

`np.column_stack([A, b])`로 확대행렬을 만든다. 먼저 두 rank가 다른지 확인한 뒤, 같을 때만 변수 수와 비교한다.
</details>

<details>
<summary>힌트 5: lstsq 반환값</summary>

`np.linalg.lstsq`는 계수, 잔차 제곱합 배열, 판정된 rank, 특이값을 순서대로 반환한다. 실제 잔차 벡터는 `b - A @ coef`로 따로 계산한다.
</details>

<details>
<summary>힌트 6: 종속 열과 같은 예측</summary>

`A_dep`의 둘째 열은 첫째 열의 2배다. 두 계수의 변화가 서로 상쇄되도록 `null_shift`를 구성하면 예측은 변하지 않는다. `solve` 시도는 `try`/`except np.linalg.LinAlgError`로 감싸도 좋다.
</details>


## 결과 해석과 마무리

- 열벡터 선형결합과 직접 행렬곱이 일치한 과정:
- 행 배치에서 전치가 필요했던 Shape 이유:
- rank가 줄었을 때 영공간과 입력 정보에 생긴 변화:
- 같은 `A`에서 `b`에 따라 무한해와 해 없음이 갈린 이유:
- `lstsq`의 네 반환값과 실제 잔차 벡터의 차이:
- `A.T @ r` 결과가 최소제곱의 최적성에 대해 보여준 것:
- 종속 열에서 예측은 같지만 계수가 여러 개가 된 이유:
- 정규방정식 `solve` 관찰 결과와 필요한 rank 조건:
- 이번 실험은 작은 정수 행렬과 정확한 종속만 사용했다. 실제 측정 노이즈나 거의 종속인 열에서는 무엇을 추가로 살펴봐야 하는가?

### 실행 검증 메모

- 이 노트북은 작은 정수 행렬과 정확히 종속된 열만 사용한다. 실제 데이터에서는 작은 특이값과 조건수도 함께 확인해야, 거의 종속된 열에서 생기는 수치적 불안정을 파악할 수 있다.
